# TEST TẠO NHÂN VẬT 3D CÓ XƯƠNG + RENDER DEMO BLENDER (COLAB PRO)
Chạy tuần tự từ trên xuống dưới (không cần cài AccuRIG hay Blender trên máy Windows):
1. Kéo code dự án từ GitHub + tải mã nguồn `Hunyuan3D-2.1`.
2. Cài thư viện `Hunyuan3D-2.1` + tải **Blender 4.2** chạy ngầm trên Colab.
3. Tạo ảnh mẫu **A-Pose** cho nhân vật `hoa_do` (hoặc dùng ảnh bạn đặt sẵn).
4. Chạy **Hunyuan3D-2.1** tạo mô hình `.glb` có màu PBR.
5. Chạy **Blender (`auto_rig_blender.py`)** tự động gắn xương, tạo hoạt ảnh đấm/xoay 360° → xuất `hoa_do_rigged.glb` + video `demo_3d_hoa_do.mp4`.
6. Tải về qua link **Gradio**.

In [ ]:
# CELL 1: KÉO DỰ ÁN & MÃ NGUỒN HUNYUAN3D-2.1
import os
if not os.path.exists("/content/Test"):
    !git clone https://github.com/nguyenkhanhan123/Test.git /content/Test
else:
    %cd /content/Test
    !git fetch origin main && git reset --hard origin/main

if not os.path.exists("/content/Hunyuan3D-2.1"):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /content/Hunyuan3D-2.1
%cd /content/Hunyuan3D-2.1

In [ ]:
# CELL 2: CÀI HUNYUAN3D-2.1 (TƯƠNG THÍCH PYTHON 3.13), BIÊN DỊCH MESH PAINTER & TẢI BLENDER 4.2
!apt-get -qq update && apt-get -qq install -y python3-dev
%cd /content/Hunyuan3D-2.1
!sed -i -E 's/^(torch|torchvision|torchaudio|numpy|bpy|pandas|basicsr|realesrgan)/# \1/' requirements.txt
!pip install -q pybind11 trimesh pymeshlab xatlas rembg onnxruntime einops omegaconf diffusers transformers accelerate opencv-python-headless scipy scikit-image
!pip install -q -r requirements.txt

%cd /content/Hunyuan3D-2.1/hy3dpaint/custom_rasterizer
!pip install -q --no-build-isolation -e .

%cd /content/Hunyuan3D-2.1/hy3dpaint/DifferentiableRenderer
!bash compile_mesh_painter.sh

if not os.path.exists("/content/blender/blender"):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
    !rm -f /content/blender.tar.xz
!/content/blender/blender --version
%cd /content/Test

In [ ]:
# CELL 3: TẠO ẢNH MẪU A-POSE (CHUẨN ĐỐI XỨNG NỀN XÁM TRƠN)
import gc, torch
from pathlib import Path
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

apose_dir = Path("/content/Test/assets/apose_inputs")
apose_dir.mkdir(parents=True, exist_ok=True)
apose_img = apose_dir / "hoa_do_apose.png"

if not apose_img.exists():
    pipe = StableDiffusionPipeline.from_pretrained("Lykon/dreamshaper-8", torch_dtype=torch.float16, safety_checker=None).to("cuda")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++", final_sigmas_type="sigma_min")
    prompt = "Full body photorealistic studio portrait of a 22-year-old Vietnamese young man wearing a fitted crimson red and black futuristic tactical jacket, dark pants and boots, standing straight facing directly forward in a symmetrical A-pose, arms relaxed at 45 degrees away from body, feet shoulder-width apart, flat even studio lighting, plain solid light gray background, sharp focus, 8k"
    neg = "blurry, cropped, sitting, side view, complex background, shadow, extra limbs, weapon, text"
    im = pipe(prompt=prompt, negative_prompt=neg, width=512, height=768, num_inference_steps=25, generator=torch.Generator("cuda").manual_seed(202601)).images[0]
    im.save(apose_img)
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
print("Ảnh A-Pose:", apose_img)

In [ ]:
# CELL 4: TẠO MÔ HÌNH 3D (.GLB) CÓ MÀU PBR TỪ ẢNH A-POSE BẰNG HUNYUAN3D-2.1
!pip install -q --upgrade transformers pymeshlab xatlas pytorch_lightning
%cd /content/Test
!git fetch origin main && git reset --hard origin/main
!rm -f /content/Test/assets/models_3d/hoa_do_raw.glb
!python -u run_hunyuan3d.py

In [ ]:
# CELL 5: TỰ ĐỘNG GẮN XƯƠNG + TẠO HOẠT ẢNH + RENDER VIDEO 3D BẰNG BLENDER
%cd /content/Test
!/content/blender/blender --background --python auto_rig_blender.py -- "/content/Test/assets/models_3d/hoa_do_raw.glb" "/content/Test/assets/models_3d/hoa_do_rigged.glb" "/content/Test/build_output/demo_3d_hoa_do.mp4" 

In [ ]:
# CELL 6: TẢI FILE 3D (.GLB) & VIDEO DEMO (.MP4) VỀ MÁY
!pip install -q gradio
import gradio as gr

with gr.Blocks() as demo:
    gr.File(value="/content/Test/assets/models_3d/hoa_do_rigged.glb", label="Model 3D có xương (hoa_do_rigged.glb)")
    gr.File(value="/content/Test/build_output/demo_3d_hoa_do.mp4", label="Video Demo Blender 3D (demo_3d_hoa_do.mp4)")

demo.launch(share=True)